In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import folium
from folium.plugins import HeatMap

In [ ]:
gdf = gpd.read_file('data/cleaned/Collision_All_Filtered.geojson')
gdf

In [ ]:
gdf['X'] = gdf.geometry.x
gdf['Y'] = gdf.geometry.y

sf_map = folium.Map(location=[gdf['Y'].mean(), gdf['X'].mean()],  zoom_start=12, control_scale=True, min_zoom=11)

gdf_filtered = gdf[(gdf["YEAR"]>= 2015)]
HeatMap(data=gdf_filtered[["X","Y","VEHCOUNT"]].groupby(['Y', 'X']).sum().reset_index().values.tolist(), radius=8, max_zoom=13).add_to(sf_map)

sf_map

In [ ]:
# Plot collisions by collision severity
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
inj_plot = gdf.plot(column='SEVERITYDESC', ax=ax, legend=True, markersize=2, categorical=True, alpha=0.7, cmap='Paired')
ax.set_title('Collision Locations Colored by Injuries (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    inj_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by number of injuries
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
inj_plot = gdf.plot(column='INJURIES', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Oranges', scheme='quantiles')
ax.set_title('Collision Locations Colored by Injuries (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    inj_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by number of fatalities
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
fat_plot = gdf.plot(column='FATALITIES', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Reds', scheme='quantiles')
ax.set_title('Collision Locations Colored by Fatalities (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    fat_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by vehicle count
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
veh_plot = gdf.plot(column='VEHCOUNT', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Blues', scheme='quantiles')
ax.set_title('Collision Locations Colored by Vehicle Count (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    veh_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by total pedestrian count
gdf['TOTAL_PED'] = gdf['PEDCOUNT'] + gdf['PEDCYLCOUNT']
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
ped_plot = gdf.plot(column='TOTAL_PED', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Greys', scheme='quantiles')
ax.set_title('Collision Locations Colored by Total Pedestrian Count (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    ped_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

gdf_int = gdf[(gdf['JUNCTIONTYPE'] == 'At Intersection (intersection related)') | (gdf['JUNCTIONTYPE'] == 'Mid-Block (but intersection related)')]

In [ ]:
gdf = gdf.dropna(subset=['geometry'])
gdf['SEVERITY'] = gdf['INJURIES'] + (gdf['SERIOUSINJURIES'] * 3) + (gdf['FATALITIES'] * 5)
df_pre = gdf[gdf['YEAR'] < 2020]
df_post = gdf[gdf['YEAR'] >= 2020]

pre_2020 = df_pre['SEVERITY']
post_2020 = df_post['SEVERITY']

print(f'{pre_2020.describe()}\n')
print(f'{post_2020.describe()}\n')

print(f'Pre-2020 mean severity per collision: {pre_2020.mean():.2f}')
print(f'Post-2020 mean severity per collision: {post_2020.mean():.2f}')

In [ ]:
gdf['lat_bin'] = pd.cut(gdf.geometry.y, bins=200)
gdf['lon_bin'] = pd.cut(gdf.geometry.x, bins=200)

grid = gdf.groupby(['lat_bin', 'lon_bin']).agg(
    collision_count=('COLDETKEY', 'count'),
    total_severity=('SEVERITY', 'sum')
).reset_index().dropna()

grid_sorted = grid.sort_values('collision_count')
grid_sorted['cum_collisions'] = grid_sorted['collision_count'].cumsum() / grid_sorted['collision_count'].sum()
grid_sorted['cum_severity'] = grid_sorted['total_severity'].cumsum() / grid_sorted['total_severity'].sum()

coords_all = np.vstack([gdf.geometry.x, gdf.geometry.y])

df_severe = gdf[(gdf['FATALITIES'] > 0) | (gdf['SERIOUSINJURIES'] > 0)]
coords_severe = np.vstack([df_severe.geometry.x, df_severe.geometry.y])

kde_all = gaussian_kde(coords_all)
kde_severe = gaussian_kde(coords_severe)

x = np.linspace(gdf.geometry.x.min(), gdf.geometry.x.max(), 200)
y = np.linspace(gdf.geometry.y.min(), gdf.geometry.y.max(), 200)
xx, yy = np.meshgrid(x, y)
grid_points = np.vstack([xx.ravel(), yy.ravel()])

z_all = kde_all(grid_points).reshape(xx.shape)
z_severe = kde_severe(grid_points).reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].contourf(xx, yy, z_all, cmap='YlOrRd', levels=20)
ax[0].set_title('KDE: All Collisions')
ax[1].contourf(xx, yy, z_severe, cmap='YlOrRd', levels=20)
ax[1].set_title('KDE: Severe Collisions Only')
plt.tight_layout()
plt.show()

In [ ]:
downtown_mask = (
    (gdf.geometry.y >= 47.595) & (gdf.geometry.y <= 47.620) &
    (gdf.geometry.x >= -122.345) & (gdf.geometry.x <= -122.320)
    )

df_downtown = gdf[downtown_mask]
df_outer = gdf[~downtown_mask]

# Count collisions per zone per year
downtown_by_year = df_downtown.groupby('YEAR').size().reset_index(name='COUNT')
outer_by_year = df_outer.groupby('YEAR').size().reset_index(name='COUNT')

downtown_area = (47.620 - 47.595) * (122.345 - 122.320)
outer_area = (gdf.geometry.y.max() - gdf.geometry.y.min()) * \
             (gdf.geometry.x.max() - gdf.geometry.x.min()) - downtown_area

downtown_by_year['DENSITY'] = downtown_by_year['COUNT'] / downtown_area
outer_by_year['DENSITY'] = outer_by_year['COUNT'] / outer_area

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, data) in zip(axes, [('Downtown', downtown_by_year), 
                                      ('Outer Areas', outer_by_year)]):
    ax.plot(data['YEAR'], data['DENSITY'], marker='o', color='steelblue')
    ax.set_title(f'{label} Collision Density Over Time', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Collisions per unit area')
    ax.legend()

plt.suptitle('Collision Density: Downtown vs Outer Areas', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summarize the pre/post change
print("Collision Density Change Post-2020:")
for label, data in [('Downtown', downtown_by_year), ('Outer', outer_by_year)]:
    pre = data[data['YEAR'] < 2020]['DENSITY'].mean()
    post = data[data['YEAR'] >= 2020]['DENSITY'].mean()
    pct_change = ((post - pre) / pre) * 100
    print(f"{label}:")
    print(f"  Pre-2020 avg density:  {pre:.2f}")
    print(f"  Post-2020 avg density: {post:.2f}")
    print(f"  % Change: {pct_change:+.1f}%")

In [ ]:
# Plot severity over time side by side
dt_sev_year = df_downtown.groupby('YEAR')['SEVERITY'].mean().reset_index()
outer_sev_year = df_outer.groupby('YEAR')['SEVERITY'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, data) in zip(axes, [('Downtown', dt_sev_year), 
                                      ('Outer Areas', outer_sev_year)]):
    ax.plot(data['YEAR'], data['SEVERITY'], marker='o', color='steelblue')
    ax.set_title(f'{label} Collision Severity Mean Over Time', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Collisions per unit area')
    ax.legend()

plt.suptitle('Collision Severity Mean: Downtown vs Outer Areas', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

df_downtown_pre = df_downtown[df_downtown['YEAR'] < 2020]['SEVERITY']
df_downtown_post = df_downtown[df_downtown['YEAR'] >= 2020]['SEVERITY']

print(f"Downtown pre-2020 mean severity:  {df_downtown_pre.mean():.4f}")
print(f"Downtown post-2020 mean severity: {df_downtown_post.mean():.4f}")
print(f"Change: {df_downtown_post.mean() - df_downtown_pre.mean():+.4f}")

# Same for outer
df_outer_pre = df_outer[df_outer['YEAR'] < 2020]['SEVERITY']
df_outer_post = df_outer[df_outer['YEAR'] >= 2020]['SEVERITY']

print(f"\nOuter pre-2020 mean severity:  {df_outer_pre.mean():.4f}")
print(f"Outer post-2020 mean severity: {df_outer_post.mean():.4f}")
print(f"Change: {df_outer_post.mean() - df_outer_pre.mean():+.4f}")

In [ ]:
gdf.to_file('data/processed/Collision_Processed.geojson', driver='GeoJSON')